# Missing Charts Analysis

Last updated by Michael Harries and Claude Code, Mar 15, 2026.

Part of the IEEE RAS robotics industry report generation pipeline.

---

Generates 19 IEEE publication-quality charts that are missing from the main notebook. Uses the same Tracxn Excel data, visual standards, and output format.

## Charts Generated

| # | Group | Chart | Output Filename |
|---|-------|-------|-----------------|
| 1 | FA Year-by-Year | Global Funding Amount | global_fa_yearly |
| 2 | FA Year-by-Year | Americas FA | americas_fa_yearly |
| 3 | FA Year-by-Year | Asia/Pacific FA | asia_pacific_fa_yearly |
| 4 | FA Year-by-Year | Europe/Africa FA | europe_africa_fa_yearly |
| 5 | FE Year-by-Year | Global Funding Events | global_fe_yearly |
| 6 | FE Year-by-Year | Americas FE | americas_fe_yearly |
| 7 | FE Year-by-Year | Asia/Pacific FE | asia_pacific_fe_yearly |
| 8 | FE Year-by-Year | Europe/Africa FE | europe_africa_fe_yearly |
| 9 | Funded-by-Year | Global Funded by Year | global_existing_funded_by_year |
| 10 | Funded-by-Year | Americas Funded by Year | americas_existing_funded_by_year |
| 11 | Funded-by-Year | USA Funded by Year | usa_existing_funded_by_year |
| 12 | Stacked-Funded | Asia/Pacific Companies by Year | asia_pacific_companies_by_year_funded |
| 13 | Stacked-Funded | Europe/Africa Companies by Year | europe_africa_companies_by_year_funded |
| 14 | Regional-% | Deadpooled by Region | regional_deadpooled_percentage |
| 15 | Regional-% | Funded by Region | regional_funded_percentage |
| 16 | Regional-Pie | Funded Companies Pie | regional_funded_companies_pie |
| 17 | Top-10 | Seed + Series A | top10_stage_SeedA |
| 18 | Top-10 | Employee Count | top10_total_employee_count |
| 19 | Global VC | VC Totals | global_vc_totals |

## Required Inputs

| File | Key columns |
|---|---|
| Companies export | `Subcategory`, `Domain Name`, `Category`, `Country`, `Founded Year`, `Is Funded`, `Is Deadpooled`, `Company Stage`, `Total Employee Count` |
| Funding rounds export | `Domain Name`, `Round Amount (in USD)` or `Round Amount (USD)`, `Round Date` |

**Optional input:** Global VC totals CSV with `Year` and `Global VC Investment (USD billions)` columns.

## Note on Seed+A

The Top 10 Seed+A chart combines companies with Company Stage "Seed" OR "Series A" (there is no literal "Seed+A" stage value).


In [ ]:
# Cell 2: Setup & Imports
!pip install -q openpyxl

%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import pandas as pd
import numpy as np
import os
import zipfile
import gc
from google.colab import files


In [ ]:
# Cell 3: Configuration

# --- Workflow Control ---
SKIP_TRACXN_UPLOAD = False
SKIP_VC_UPLOAD = False

# Year ranges
START_YEAR_ALL = 1900
START_YEAR_DEFAULT = 2000
FUNDED_START_YEAR = 2011

# Funding thresholds
ROUND_SIZE_THRESHOLD = 100_000_000  # $100M boundary for stacked funding charts

# Display limits
TOP_SECTORS_COUNT = 7  # Sectors shown individually before 'Other' bucket

# Output
CHART_OUTPUT_DIR = 'no_title_charts'

# --- Color Palette (Paul Tol Colorblind-Friendly) ---
PAUL_TOL_PRIMARY = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377', '#BBBBBB']

PAUL_TOL_EXTENDED = PAUL_TOL_PRIMARY + [
    '#332288', '#88CCEE', '#44AA99', '#117733',
    '#999933', '#DDCC77', '#CC6677', '#882255', '#AA4499'
]

BINARY_STATUS_COLORS = ['#88CCEE', '#332288']

# --- Country Standardization ---
COUNTRY_STANDARDIZATION = {
    'USA': 'United States',
    'Republic of Korea': 'South Korea',
    'United Arab Emirates': 'UAE',
}

# --- Second-Tier Region Mapping ---
SECOND_TIER_REGIONS = {
    'USA': ['United States'],
    'Canada': ['Canada'],
    'China': ['China'],
    'United Kingdom': ['United Kingdom'],
    'India': ['India'],
    'APAC': [
        'Japan', 'Kazakhstan', 'Nepal', 'Pakistan', 'Australia',
        'Bangladesh', 'New Zealand', 'South Korea', 'Singapore',
        'Indonesia', 'Malaysia', 'Thailand', 'Vietnam',
        'Philippines', 'Sri Lanka', 'Taiwan'
    ],
    'Middle East': [
        'Jordan', 'Kuwait', 'Oman', 'Armenia', 'Israel',
        'Turkey', 'UAE', 'Iran', 'Lebanon', 'Saudi Arabia'
    ],
    'Africa': [
        'Senegal', 'Rwanda', 'Ghana', 'Tunisia', 'Morocco',
        'South Africa', 'Egypt', 'Nigeria', 'Kenya'
    ],
    'Western Europe': [
        'Germany', 'France', 'Spain', 'Italy', 'Netherlands',
        'Austria', 'Belgium', 'Switzerland', 'Luxembourg',
        'Portugal', 'Ireland'
    ],
    'Eastern Europe': [
        'Belarus', 'Serbia', 'Bulgaria', 'Ukraine', 'Poland',
        'Russia', 'Czech Republic', 'Slovakia', 'Slovenia',
        'Croatia', 'Cyprus', 'Greece', 'Hungary', 'Romania'
    ],
    'Nordics and Baltics': [
        'Sweden', 'Norway', 'Finland', 'Denmark', 'Iceland',
        'Estonia', 'Latvia', 'Lithuania'
    ],
    'Americas (No Canada and USA)': [
        'Guatemala', 'Puerto Rico', 'Costa Rica', 'Uruguay',
        'Panama', 'Ecuador', 'Mexico', 'Brazil', 'Argentina',
        'Chile', 'Colombia', 'Peru'
    ],
}

# --- Reverse Region Lookup ---
COUNTRY_TO_REGION = {}
for region, countries in SECOND_TIER_REGIONS.items():
    for country in countries:
        COUNTRY_TO_REGION[country] = region

# Macro-region grouping
MACRO_REGION_GROUPS = {
    'Americas': ['USA', 'Canada', 'Americas (No Canada and USA)'],
    'Europe/Africa': ['United Kingdom', 'Western Europe', 'Eastern Europe',
                      'Nordics and Baltics', 'Middle East', 'Africa'],
    'Asia/Pacific': ['China', 'India', 'APAC'],
}

COUNTRY_TO_MACRO_REGION = {}
for macro, second_tiers in MACRO_REGION_GROUPS.items():
    for st in second_tiers:
        for country in SECOND_TIER_REGIONS[st]:
            COUNTRY_TO_MACRO_REGION[country] = macro


In [ ]:
# Cell 4: Data Upload & Cleaning
import re
from datetime import datetime

# --- Cleaning Functions ---

def clean_amount(amount):
    """Clean and convert amount to float. Returns 0.0 on failure."""
    if pd.isna(amount) or amount is None:
        return 0.0
    if isinstance(amount, (int, float)):
        return float(amount)
    if isinstance(amount, str):
        amount = amount.replace(',', '').replace('$', '').replace(' ', '')
        if amount == '':
            return 0.0
        try:
            return float(amount)
        except ValueError:
            return 0.0
    return 0.0

def extract_year(date_str):
    """Extract year from various date formats. Valid range: 1800-2030."""
    if pd.isna(date_str) or date_str is None:
        return None
    date_str = str(date_str).strip()
    if date_str == '' or date_str.lower() == 'none':
        return None

    # Handle bare year (4 digits)
    if len(date_str) == 4 and date_str.isdigit():
        year = int(date_str)
        return year if 1800 <= year <= 2030 else None

    # Handle "Jan 01, 2020" format (Tracxn)
    if ',' in date_str:
        try:
            year = datetime.strptime(date_str.strip(), '%b %d, %Y').year
            return year if 1800 <= year <= 2030 else None
        except (ValueError, TypeError):
            pass

    # Try standard date formats
    for fmt in ['%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%b %Y', '%B %Y']:
        try:
            year = datetime.strptime(date_str, fmt).year
            return year if 1800 <= year <= 2030 else None
        except ValueError:
            continue

    # Regex fallback
    match = re.search(r'\b(18|19|20)\d{2}\b', date_str)
    if match:
        year = int(match.group())
        return year if 1800 <= year <= 2030 else None

    return None

def clean_category(category):
    """Clean category names - take first if comma-separated."""
    if pd.isna(category) or category is None:
        return 'Unknown'
    category = str(category).strip()
    if category == '':
        return 'Unknown'
    if ',' in category:
        category = category.split(',')[0].strip()
    return category

def clean_country(country):
    """Clean country name - take first if comma-separated, apply standardization."""
    if pd.isna(country) or country is None:
        return 'Unknown'
    country = str(country).strip()
    if country == '':
        return 'Unknown'
    if ',' in country:
        country = country.split(',')[0].strip()
    return COUNTRY_STANDARDIZATION.get(country, country)

if SKIP_TRACXN_UPLOAD:
    print("Skipping Tracxn file upload (SKIP_TRACXN_UPLOAD = True).")
    print("Only the global VC totals chart will be available.")
else:
    # --- File Upload ---
    print("Upload two Excel files: one companies export and one funding rounds export from Tracxn.")
    uploaded = files.upload()
    filenames = list(uploaded.keys())
    assert len(filenames) == 2, f"Expected 2 files, got {len(filenames)}: {filenames}"

    dfs = {}
    for fn in filenames:
        dfs[fn] = pd.read_excel(fn, engine='openpyxl')
        print(f"  Loaded {fn}: {dfs[fn].shape[0]} rows, {dfs[fn].shape[1]} columns")

    # --- Auto-Detection ---
    companies_file = None
    funding_file = None

    for fn, df in dfs.items():
        if 'Subcategory' in df.columns:
            if companies_file is not None:
                raise ValueError("Both files contain 'Subcategory'. Please upload one companies file and one funding rounds file.")
            companies_file = fn
        else:
            funding_file = fn

    if companies_file is None:
        all_cols = {fn: list(df.columns) for fn, df in dfs.items()}
        raise ValueError(f"Neither uploaded file contains a 'Subcategory' column. "
                         f"The companies file from Tracxn must include this column. "
                         f"Available columns: {all_cols}")

    if funding_file is None:
        funding_file = [fn for fn in filenames if fn != companies_file][0]

    print(f"\nDetected companies file: {companies_file}")
    print(f"Detected funding file:  {funding_file}")

    companies_raw = dfs[companies_file]
    funding_raw = dfs[funding_file]

    # --- Validation ---
    assert 'Domain Name' in companies_raw.columns, \
        f"Companies file missing 'Domain Name'. Columns: {list(companies_raw.columns)}"
    assert 'Domain Name' in funding_raw.columns, \
        f"Funding file missing 'Domain Name'. Columns: {list(funding_raw.columns)}"

    # Round amount column may be named differently across Tracxn export versions
    ROUND_AMOUNT_COL = None
    for candidate in ['Round Amount (in USD)', 'Round Amount (USD)']:
        if candidate in funding_raw.columns:
            ROUND_AMOUNT_COL = candidate
            break
    assert ROUND_AMOUNT_COL is not None, \
        f"Funding file missing round amount column. Looked for 'Round Amount (in USD)' or 'Round Amount (USD)'. Columns: {list(funding_raw.columns)}"
    print(f"Using round amount column: '{ROUND_AMOUNT_COL}'")

    assert 'Round Date' in funding_raw.columns, \
        f"Funding file missing 'Round Date'. Columns: {list(funding_raw.columns)}"

    # --- Inner Join ---
    merged_df = pd.merge(companies_raw, funding_raw, on='Domain Name', how='inner',
                         suffixes=('_Companies', '_Funding'))
    print(f"\nMerged: {merged_df.shape[0]} rows (inner join on Domain Name)")

    # --- Apply Cleaning ---
    cat_col = 'Category_Companies' if 'Category_Companies' in merged_df.columns else 'Category'
    country_col = 'Country_Companies' if 'Country_Companies' in merged_df.columns else 'Country'

    merged_df['amount_usd'] = merged_df[ROUND_AMOUNT_COL].apply(clean_amount)
    merged_df['year'] = merged_df['Round Date'].apply(extract_year)
    merged_df['category_clean'] = merged_df[cat_col].apply(clean_category)
    merged_df['country_clean'] = merged_df[country_col].apply(clean_country)

    # --- Filter to analysis_df (disclosed amounts, valid years) ---
    analysis_df = merged_df[(merged_df['amount_usd'] > 0) & (merged_df['year'].notna())].copy()
    analysis_df['year'] = analysis_df['year'].astype(int)

    # --- all_events_df (all events with valid years, including undisclosed) ---
    all_events_df = merged_df[merged_df['year'].notna()].copy()
    all_events_df['year'] = all_events_df['year'].astype(int)

    # --- companies_df (companies-only, for founded-year charts) ---
    companies_df = companies_raw.copy()
    companies_df['founded_year'] = companies_df['Founded Year'].apply(extract_year)
    companies_df = companies_df[companies_df['founded_year'].notna()].copy()
    companies_df['founded_year'] = companies_df['founded_year'].astype(int)
    companies_df = companies_df[(companies_df['founded_year'] >= 1800) & (companies_df['founded_year'] <= 2030)]
    companies_df['category_clean'] = companies_df['Category'].apply(clean_category)
    companies_df['country_clean'] = companies_df['Country'].apply(clean_country)

    # --- Summary Statistics ---
    print(f"\n{'='*50}")
    print(f"DATA PIPELINE SUMMARY")
    print(f"{'='*50}")
    print(f"Companies uploaded:    {companies_raw.shape[0]:,}")
    print(f"Funding rounds uploaded: {funding_raw.shape[0]:,}")
    print(f"Merged rows:           {merged_df.shape[0]:,}")
    print(f"analysis_df rows:      {analysis_df.shape[0]:,} (disclosed amounts, valid years)")
    print(f"all_events_df rows:    {all_events_df.shape[0]:,} (all events, valid years)")
    print(f"companies_df rows:     {companies_df.shape[0]:,} (valid founded year)")
    print(f"Year range:            {int(analysis_df['year'].min())} - {int(analysis_df['year'].max())}")
    print(f"Unique companies:      {merged_df['Domain Name'].nunique():,}")
    print(f"{'='*50}")


In [ ]:
# Cell 5: Shared Utilities

def setup_charts():
    """Configure matplotlib rcParams for IEEE-compatible styling in Colab."""
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif', 'Liberation Serif', 'serif'],
        'text.usetex': False,
        'font.size': 10,
        'axes.labelsize': 10,
        'axes.titlesize': 12,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'figure.dpi': 100,
        'savefig.dpi': 300,
        'figure.figsize': [7.25, 4.48],
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.8,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.5,
        'savefig.bbox': 'tight',
        'savefig.pad_inches': 0.05,
        'savefig.facecolor': 'white',
        'savefig.edgecolor': 'none',
        'lines.linewidth': 1.5,
        'lines.markersize': 6,
    })
    plt.rcParams['axes.edgecolor'] = '#CCCCCC'

setup_charts()


def format_currency_axis(ax, axis='y', scale='auto'):
    """Apply currency formatting to an axis."""
    if scale == 'auto':
        if axis == 'y':
            lim = ax.get_ylim()
        else:
            lim = ax.get_xlim()
        max_val = max(abs(lim[0]), abs(lim[1]))
        scale = 'B' if max_val >= 1.0 else 'M'
    if scale == 'B':
        formatter = FuncFormatter(lambda x, p: f'${x:.1f}B')
    else:
        formatter = FuncFormatter(lambda x, p: f'${x:.0f}M')
    if axis == 'y':
        ax.yaxis.set_major_formatter(formatter)
    else:
        ax.xaxis.set_major_formatter(formatter)


def save_figure(fig, filename):
    """Save figure as PNG to CHART_OUTPUT_DIR with _no_title suffix."""
    os.makedirs(CHART_OUTPUT_DIR, exist_ok=True)
    filepath = os.path.join(CHART_OUTPUT_DIR, f'{filename}_no_title.png')
    fig.savefig(filepath, dpi=300, bbox_inches='tight', pad_inches=0.05)
    print(f'Saved: {filepath}')


def get_colors(n):
    """Return n colors from Paul Tol colorblind-friendly palette."""
    if n <= 7:
        return PAUL_TOL_PRIMARY[:n]
    elif n <= 16:
        return PAUL_TOL_EXTENDED[:n]
    else:
        return [PAUL_TOL_EXTENDED[i % len(PAUL_TOL_EXTENDED)] for i in range(n)]


def sanitize_filename(name):
    """Convert display name to filename-safe string."""
    import re as _re
    name = name.replace('&', 'and')
    name = name.replace('/', '_')
    name = name.replace(',', '')
    name = name.replace('(', '').replace(')', '')
    name = name.replace(' ', '_')
    name = _re.sub(r'[^a-zA-Z0-9_.]', '', name)
    return name


def complete_year_range(data, start_year=None, end_year=None, year_col=None):
    """Ensure data has entries for every year in range, filling gaps with 0."""
    if isinstance(data, pd.Series):
        if start_year is None:
            start_year = int(data.index.min())
        if end_year is None:
            end_year = int(data.index.max())
        full_range = range(start_year, end_year + 1)
        return data.reindex(full_range, fill_value=0)
    else:
        if year_col is None:
            raise ValueError("year_col required for DataFrame input")
        if start_year is None:
            start_year = int(data[year_col].min())
        if end_year is None:
            end_year = int(data[year_col].max())
        full_range = pd.DataFrame({year_col: range(start_year, end_year + 1)})
        return full_range.merge(data, on=year_col, how='left').fillna(0)


def is_yes(series):
    """Check if a pandas Series contains 'Yes' values (case-insensitive)."""
    return series.astype(str).str.strip().str.lower() == 'yes'


In [ ]:
# Cell 6: Chart Function Definitions

# --- 1. Funding Amount / Funding Events Year-by-Year ---

def create_funding_yearly_bar(df, metric, scope_filter, start_year, filename):
    """Simple single-series bar chart for funding amount or event count by year.

    Args:
        df: analysis_df (metric='amount') or all_events_df (metric='count')
        metric: 'amount' for total funding, 'count' for event count
        scope_filter: None or callable(df) -> filtered df
        start_year: filter to years >= start_year
        filename: output filename stem
    """
    filtered = scope_filter(df) if scope_filter else df
    filtered = filtered[filtered['year'] >= start_year]
    if len(filtered) == 0:
        return None

    if metric == 'amount':
        yearly = filtered.groupby('year')['amount_usd'].sum()
    else:
        yearly = filtered.groupby('year').size()

    yearly = complete_year_range(yearly, start_year=start_year)

    fig, ax = plt.subplots()
    color = PAUL_TOL_PRIMARY[0]  # #4477AA

    if metric == 'amount':
        yearly_scaled = yearly / 1e9
        ax.bar(yearly_scaled.index, yearly_scaled.values, color=color,
               edgecolor='black', linewidth=0.3, width=0.8)
        format_currency_axis(ax, axis='y', scale='auto')
        ax.set_ylabel('Total Funding Amount (USD)')

        # Value labels on last 5 bars
        n_labels = min(5, len(yearly_scaled))
        for idx in yearly_scaled.index[-n_labels:]:
            val = yearly_scaled[idx]
            if val >= 1.0:
                label = f'${val:.1f}B'
            else:
                label = f'${val*1000:.0f}M'
            ax.text(idx, val, label, ha='center', va='bottom', fontsize=7, fontweight='bold')
    else:
        ax.bar(yearly.index, yearly.values, color=color,
               edgecolor='black', linewidth=0.3, width=0.8)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_ylabel('Number of Funding Events')

        # Value labels on last 6 bars
        n_labels = min(6, len(yearly))
        for idx in yearly.index[-n_labels:]:
            val = int(yearly[idx])
            ax.text(idx, val, f'{val:,}', ha='center', va='bottom', fontsize=7, fontweight='bold')

    ax.set_xlabel('Year')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 2. Companies Founded by Year ---

def create_companies_founded_by_year(companies_df, scope_filter, filename):
    """Bar chart of all companies by founded year.

    Counts all companies grouped by founded year, bucketed into
    '< 2000s', '2000 - 2010', then individual years from 2011 onward.

    Args:
        companies_df: company-level DataFrame with 'founded_year'
        scope_filter: None or callable(df) -> filtered df
        filename: output filename stem
    """
    filtered = scope_filter(companies_df) if scope_filter else companies_df
    if len(filtered) == 0:
        return None

    max_year = int(filtered['founded_year'].max())

    def year_bucket(y):
        if y < 2000:
            return '< 2000s'
        elif y <= 2010:
            return '2000 - 2010'
        else:
            return str(int(y))

    df = filtered.copy()
    df['year_bucket'] = df['founded_year'].apply(year_bucket)

    bucket_order = ['< 2000s', '2000 - 2010'] + [str(y) for y in range(2011, max_year + 1)]
    bucket_order = [b for b in bucket_order if b in df['year_bucket'].unique()]

    counts = df.groupby('year_bucket').size()
    counts = counts.reindex(bucket_order, fill_value=0)

    fig, ax = plt.subplots()
    color = BINARY_STATUS_COLORS[1]  # #332288
    x = range(len(counts))
    bars = ax.bar(x, counts.values, color=color, edgecolor='black', linewidth=0.3, width=0.8)

    # Value labels on ALL bars
    for i, v in enumerate(counts.values):
        if v > 0:
            ax.text(i, v, f'{int(v)}', ha='center', va='bottom', fontsize=7, fontweight='bold')

    ax.set_xticks(list(x))
    ax.set_xticklabels(counts.index, rotation=45, ha='right')
    ax.set_ylabel('Number of Companies')
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 3. Companies by Year Funded (Stacked Yes/No) ---

def create_companies_by_year_funded_stacked(companies_df, scope_filter, start_year, filename):
    """Stacked bar chart: companies by founded year, split by funded Yes/No.

    Args:
        companies_df: company-level DataFrame
        scope_filter: None or callable(df) -> filtered df
        start_year: first individual year to show (years before are bucketed)
        filename: output filename stem
    """
    filtered = scope_filter(companies_df) if scope_filter else companies_df
    if len(filtered) == 0:
        return None

    df = filtered.copy()
    df['is_funded'] = is_yes(df['Is Funded'])
    max_year = int(df['founded_year'].max())

    def year_bucket(y):
        if y < 2000:
            return '< 2000s'
        elif y <= 2010:
            return '2000 - 2010'
        else:
            return str(int(y))

    df['year_bucket'] = df['founded_year'].apply(year_bucket)

    bucket_order = ['< 2000s', '2000 - 2010'] + [str(y) for y in range(start_year, max_year + 1)]
    bucket_order = [b for b in bucket_order if b in df['year_bucket'].unique()]

    funded_yes = df[df['is_funded']].groupby('year_bucket').size()
    funded_no = df[~df['is_funded']].groupby('year_bucket').size()
    funded_yes = funded_yes.reindex(bucket_order, fill_value=0)
    funded_no = funded_no.reindex(bucket_order, fill_value=0)

    fig, ax = plt.subplots()
    x = range(len(bucket_order))
    width = 0.8

    ax.bar(x, funded_no.values, width, color=BINARY_STATUS_COLORS[0], label='No',
           edgecolor='black', linewidth=0.3)
    ax.bar(x, funded_yes.values, width, bottom=funded_no.values, color=BINARY_STATUS_COLORS[1],
           label='Yes', edgecolor='black', linewidth=0.3)

    # Value labels inside segments
    for i in range(len(bucket_order)):
        if funded_no.values[i] > 0:
            ax.text(i, funded_no.values[i] / 2, f'{int(funded_no.values[i])}',
                    ha='center', va='center', fontsize=7, color='black')
        if funded_yes.values[i] > 0:
            ax.text(i, funded_no.values[i] + funded_yes.values[i] / 2,
                    f'{int(funded_yes.values[i])}',
                    ha='center', va='center', fontsize=7, color='white')

    ax.set_xticks(list(x))
    ax.set_xticklabels(bucket_order, rotation=45, ha='right')
    ax.set_ylabel('Number of Companies')
    ax.legend(title='Funded?', loc='upper left', fontsize=9)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 4. Regional Status Percentage Bars ---

def create_regional_status_bars(companies_df, metric, filename):
    """100% stacked bar chart by macro-region for deadpooled or funded status.

    Args:
        companies_df: company-level DataFrame
        metric: 'deadpooled' or 'funded'
        filename: output filename stem
    """
    df = companies_df.copy()
    df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    df = df.dropna(subset=['macro_region'])
    if len(df) == 0:
        return None

    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']

    if metric == 'deadpooled':
        df['status'] = is_yes(df['Is Deadpooled'])
        legend_title = 'Deadpooled?'
    elif metric == 'funded':
        df['status'] = is_yes(df['Is Funded'])
        legend_title = 'Funded?'
    else:
        return None

    total = df.groupby('macro_region').size()
    yes_count = df[df['status']].groupby('macro_region').size()
    no_count = total - yes_count.reindex(total.index, fill_value=0)
    yes_count = yes_count.reindex(total.index, fill_value=0)

    pct_yes = (yes_count / total * 100).reindex(region_order, fill_value=0)
    pct_no = (no_count / total * 100).reindex(region_order, fill_value=0)

    fig, ax = plt.subplots()
    x = range(len(region_order))
    width = 0.6

    ax.bar(x, pct_no.values, width, color=BINARY_STATUS_COLORS[0], label='No',
           edgecolor='black', linewidth=0.3)
    ax.bar(x, pct_yes.values, width, bottom=pct_no.values, color=BINARY_STATUS_COLORS[1],
           label='Yes', edgecolor='black', linewidth=0.3)

    # Percentage labels inside segments (white text)
    for i, region in enumerate(region_order):
        no_val = pct_no[region]
        yes_val = pct_yes[region]
        if no_val > 5:
            ax.text(i, no_val / 2, f'{no_val:.0f}%', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
        if yes_val > 5:
            ax.text(i, no_val + yes_val / 2, f'{yes_val:.0f}%', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')

    ax.set_xticks(list(x))
    ax.set_xticklabels(region_order)
    ax.set_ylabel('Percentage (%)')
    ax.set_ylim(0, 100)
    ax.legend(title=legend_title, loc='upper right', fontsize=9)
    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 5. Funded Companies Regional Pie ---

def create_funded_region_pie(companies_df, filename):
    """Pie chart of funded companies distribution by macro-region.

    Args:
        companies_df: company-level DataFrame
        filename: output filename stem
    """
    df = companies_df.copy()
    df = df[is_yes(df['Is Funded'])]
    df['macro_region'] = df['country_clean'].map(COUNTRY_TO_MACRO_REGION)
    df = df.dropna(subset=['macro_region'])
    if len(df) == 0:
        return None

    region_order = ['Americas', 'Europe/Africa', 'Asia/Pacific']
    counts = df.groupby('macro_region').size().reindex(region_order, fill_value=0)

    fig, ax = plt.subplots()
    colors = get_colors(3)

    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=None,
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.65,
        wedgeprops=dict(edgecolor='black', linewidth=0.5)
    )
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(11)

    ax.legend(region_order, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
    ax.set_aspect('equal')
    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 6. Top 10 Countries Chart ---

def create_top10_countries_chart(df, metric_col, metric_label, filename, color=None):
    """Vertical bar chart: top 10 countries by a given metric.

    Args:
        df: DataFrame with 'country_clean' and the metric column
        metric_col: column name to aggregate (or 'count' for row count)
        metric_label: y-axis label
        filename: output filename stem
        color: optional bar color override
    """
    if metric_col == 'count':
        totals = df.groupby('country_clean').size().sort_values(ascending=False)
    else:
        totals = df.groupby('country_clean')[metric_col].sum().sort_values(ascending=False)

    top10 = totals.head(10)
    if len(top10) == 0:
        return None

    fig, ax = plt.subplots()
    bar_color = color if color else get_colors(1)[0]
    x = range(len(top10))
    ax.bar(x, top10.values, color=bar_color, edgecolor='black', linewidth=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(top10.index, rotation=45, ha='right')
    ax.set_ylabel(metric_label)

    for i, v in enumerate(top10.values):
        ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    plt.tight_layout()
    save_figure(fig, filename)
    return fig


# --- 7. Global VC Funding Chart ---

def create_global_vc_funding_chart(vc_df, start_year, filename):
    """Bar chart of global VC investment by year from summary CSV.

    Args:
        vc_df: DataFrame with 'Year' and 'Global VC Investment (USD billions)' columns
        start_year: first year to display
        filename: output filename stem
    """
    df = vc_df[vc_df['Year'] >= start_year].copy()
    df = df.sort_values('Year')

    funding = pd.Series(
        df['Global VC Investment (USD billions)'].values,
        index=df['Year'].values
    )
    funding = complete_year_range(funding, start_year=start_year)

    fig, ax = plt.subplots()
    color = get_colors(1)[0]
    ax.bar(funding.index, funding.values, color=color, edgecolor='black',
           linewidth=0.5, alpha=0.8)
    format_currency_axis(ax, axis='y', scale='B')
    ax.set_xlabel('Year')
    ax.set_ylabel('Global VC Investment')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_figure(fig, filename)
    return fig

In [ ]:
# Cell 7: Master Generation Loop

import re

os.makedirs(CHART_OUTPUT_DIR, exist_ok=True)
charts_generated = []
charts_failed = []

# Chart descriptions for .txt metadata files
CHART_DESCRIPTIONS = {}

def generate_chart(name, chart_fn, *args, description='', **kwargs):
    """Wrapper: call chart function, track success/failure, manage memory."""
    if description:
        CHART_DESCRIPTIONS[name] = description
    try:
        fig = chart_fn(*args, **kwargs)
        if fig is not None:
            charts_generated.append(name)
            plt.show()
            plt.close(fig)
        gc.collect()
    except Exception as e:
        charts_failed.append((name, str(e)))
        print(f"  ERROR generating {name}: {e}")
        plt.close('all')
        gc.collect()

if SKIP_TRACXN_UPLOAD:
    print("Skipping Tracxn chart generation (SKIP_TRACXN_UPLOAD = True).\n")
else:
    # --- Scope Filter Definitions ---
    americas_countries = [c for st in MACRO_REGION_GROUPS['Americas'] for c in SECOND_TIER_REGIONS[st]]
    asia_pacific_countries = [c for st in MACRO_REGION_GROUPS['Asia/Pacific'] for c in SECOND_TIER_REGIONS[st]]
    europe_africa_countries = [c for st in MACRO_REGION_GROUPS['Europe/Africa'] for c in SECOND_TIER_REGIONS[st]]

    scope_americas = lambda df: df[df['country_clean'].isin(americas_countries)]
    scope_asia_pacific = lambda df: df[df['country_clean'].isin(asia_pacific_countries)]
    scope_europe_africa = lambda df: df[df['country_clean'].isin(europe_africa_countries)]
    scope_usa = lambda df: df[df['country_clean'] == 'United States']

    # === Section 1: Funding Amount Year-by-Year (Charts 1-4) ===
    print("Generating FA year-by-year charts...")
    count_before = len(charts_generated)

    generate_chart('global_fa_yearly',
        create_funding_yearly_bar, analysis_df, 'amount', None, FUNDED_START_YEAR, 'global_fa_yearly',
        description='Global total funding amount by year (2011+). Sum of disclosed funding round amounts across all countries.\nOriginal: missing/Economic Plots/NEW/Global FA - year by year.png')
    generate_chart('americas_fa_yearly',
        create_funding_yearly_bar, analysis_df, 'amount', scope_americas, FUNDED_START_YEAR, 'americas_fa_yearly',
        description='Americas total funding amount by year (2011+). Sum of disclosed funding round amounts for companies in Americas region.\nOriginal: missing/Economic Plots/NEW/Americas FA - yby.png')
    generate_chart('asia_pacific_fa_yearly',
        create_funding_yearly_bar, analysis_df, 'amount', scope_asia_pacific, FUNDED_START_YEAR, 'asia_pacific_fa_yearly',
        description='Asia/Pacific total funding amount by year (2011+). Sum of disclosed funding round amounts for companies in Asia/Pacific region.\nOriginal: missing/Economic Plots/NEW/AP FA - yby.png')
    generate_chart('europe_africa_fa_yearly',
        create_funding_yearly_bar, analysis_df, 'amount', scope_europe_africa, FUNDED_START_YEAR, 'europe_africa_fa_yearly',
        description='Europe/Africa total funding amount by year (2011+). Sum of disclosed funding round amounts for companies in Europe/Africa region.\nOriginal: missing/Economic Plots/NEW/EA FA - yby.png')

    plt.close('all')
    gc.collect()
    print(f"  FA charts: {len(charts_generated) - count_before} generated")

    # === Section 2: Funding Events Year-by-Year (Charts 5-8) ===
    print("\nGenerating FE year-by-year charts...")
    count_before = len(charts_generated)

    generate_chart('global_fe_yearly',
        create_funding_yearly_bar, all_events_df, 'count', None, FUNDED_START_YEAR, 'global_fe_yearly',
        description='Global count of funding events by year (2011+). Includes all funding rounds (disclosed and undisclosed amounts).\nOriginal: missing/Economic Plots/NEW/global FE-year by year.png')
    generate_chart('americas_fe_yearly',
        create_funding_yearly_bar, all_events_df, 'count', scope_americas, FUNDED_START_YEAR, 'americas_fe_yearly',
        description='Americas count of funding events by year (2011+). Includes all funding rounds for companies in Americas region.\nOriginal: missing/Economic Plots/NEW/Americas FE - year by year.png')
    generate_chart('asia_pacific_fe_yearly',
        create_funding_yearly_bar, all_events_df, 'count', scope_asia_pacific, FUNDED_START_YEAR, 'asia_pacific_fe_yearly',
        description='Asia/Pacific count of funding events by year (2011+). Includes all funding rounds for companies in Asia/Pacific region.\nOriginal: missing/Economic Plots/NEW/AP FE - yby.png')
    generate_chart('europe_africa_fe_yearly',
        create_funding_yearly_bar, all_events_df, 'count', scope_europe_africa, FUNDED_START_YEAR, 'europe_africa_fe_yearly',
        description='Europe/Africa count of funding events by year (2011+). Includes all funding rounds for companies in Europe/Africa region.\nOriginal: missing/Economic Plots/NEW/EA FE - yby.png')

    plt.close('all')
    gc.collect()
    print(f"  FE charts: {len(charts_generated) - count_before} generated")

    # === Section 3: Companies Founded by Year (Charts 9-11) ===
    # All companies grouped by founded year — no funding filter
    print("\nGenerating companies-founded-by-year charts...")
    count_before = len(charts_generated)

    generate_chart('global_companies_founded_by_year',
        create_companies_founded_by_year, companies_df, None, 'global_companies_founded_by_year',
        description='Global count of all companies by year founded. Bucketed into <2000s, 2000-2010, then individual years 2011+. No funding filter applied.\nOriginal: missing/Economic Plots/StateofMarket/Global - Existing Coy Funded by year.png')
    generate_chart('americas_companies_founded_by_year',
        create_companies_founded_by_year, companies_df, scope_americas, 'americas_companies_founded_by_year',
        description='Americas count of all companies by year founded. Bucketed into <2000s, 2000-2010, then individual years 2011+. No funding filter applied.\nOriginal: missing/Economic Plots/StateofMarket/Americas - Existing Coy Funded by year.png')
    generate_chart('usa_companies_founded_by_year',
        create_companies_founded_by_year, companies_df, scope_usa, 'usa_companies_founded_by_year',
        description='USA count of all companies by year founded. Bucketed into <2000s, 2000-2010, then individual years 2011+. No funding filter applied.\nOriginal: missing/Economic Plots/StateofMarket/USA - Existing Coy Funded by year.png')

    plt.close('all')
    gc.collect()
    print(f"  Companies-founded charts: {len(charts_generated) - count_before} generated")

    # === Section 4: Stacked Funded Companies by Year (Charts 12-13) ===
    print("\nGenerating stacked funded charts...")
    count_before = len(charts_generated)

    generate_chart('asia_pacific_companies_by_year_funded',
        create_companies_by_year_funded_stacked, companies_df, scope_asia_pacific, 2011, 'asia_pacific_companies_by_year_funded',
        description='Asia/Pacific companies by founded year, stacked by funded status (Yes/No). Bucketed into <2000s, 2000-2010, then individual years 2011+.\nOriginal: missing/Economic Plots/StateofMarket/AP/AP_nocoby.png')
    generate_chart('europe_africa_companies_by_year_funded',
        create_companies_by_year_funded_stacked, companies_df, scope_europe_africa, 2011, 'europe_africa_companies_by_year_funded',
        description='Europe/Africa companies by founded year, stacked by funded status (Yes/No). Bucketed into <2000s, 2000-2010, then individual years 2011+.\nOriginal: missing/Economic Plots/StateofMarket/EA/EA_nocoby.png')

    plt.close('all')
    gc.collect()
    print(f"  Stacked funded charts: {len(charts_generated) - count_before} generated")

    # === Section 5: Regional Status Percentage Bars (Charts 14-15) ===
    print("\nGenerating regional status charts...")
    count_before = len(charts_generated)

    generate_chart('regional_deadpooled_percentage',
        create_regional_status_bars, companies_df, 'deadpooled', 'regional_deadpooled_percentage',
        description='100% stacked bar chart showing deadpooled percentage by macro-region (Americas, Europe/Africa, Asia/Pacific).\nOriginal: missing/Economic Plots/StateofMarket/Reg-n-deadpcompanies.png')
    generate_chart('regional_funded_percentage',
        create_regional_status_bars, companies_df, 'funded', 'regional_funded_percentage',
        description='100% stacked bar chart showing funded percentage by macro-region (Americas, Europe/Africa, Asia/Pacific).\nOriginal: missing/Economic Plots/StateofMarket/Reg-n-fundedpcompanies.png')

    plt.close('all')
    gc.collect()
    print(f"  Regional status charts: {len(charts_generated) - count_before} generated")

    # === Section 6: Funded Companies Regional Pie (Chart 16) ===
    print("\nGenerating funded region pie chart...")
    count_before = len(charts_generated)

    generate_chart('regional_funded_companies_pie',
        create_funded_region_pie, companies_df, 'regional_funded_companies_pie',
        description='Pie chart showing distribution of funded companies across macro-regions (Americas, Europe/Africa, Asia/Pacific). Only companies with Is Funded=Yes.\nOriginal: missing/Economic Plots/StateofMarket/Reg-perc-fundedpcompanies.png')

    plt.close('all')
    gc.collect()
    print(f"  Funded pie charts: {len(charts_generated) - count_before} generated")

    # === Section 7: Top 10 Country Charts (Charts 17-18) ===
    print("\nGenerating top 10 country charts...")
    count_before = len(charts_generated)

    # Seed+A: combine Seed + Series A (not a literal stage value)
    stage_col = None
    for candidate in ['Company Stage', 'Company Stage_Companies']:
        if candidate in companies_df.columns:
            stage_col = candidate
            break

    if stage_col is not None:
        seed_a_df = companies_df[companies_df[stage_col].str.strip().isin(['Seed', 'Series A'])]
        if len(seed_a_df) > 0:
            generate_chart('top10_stage_SeedA',
                create_top10_countries_chart, seed_a_df, 'count',
                'Number of Companies', 'top10_stage_SeedA', color='#332288',
                description='Top 10 countries by number of companies at Seed or Series A stage. Combines both stages (no literal "Seed+A" stage value exists).\nOriginal: missing/Economic Plots/StateofMarket/Top 10/Top10_Stage_Seed+A.png')

    # Total Employee Count — parse K/M multipliers (e.g. "1.8K (Dec 31, 2023)")
    emp_col = None
    for candidate in ['Total Employee Count', 'Total Employee Count_Companies']:
        if candidate in companies_df.columns:
            emp_col = candidate
            break

    if emp_col is not None:
        emp_df = companies_df[companies_df[emp_col].notna()].copy()

        def parse_employee_count(val):
            if pd.isna(val):
                return None
            s = str(val).replace('+', '').replace(',', '').strip()
            m = re.match(r'([\d.]+)\s*([KkMm])?', s)
            if not m:
                return None
            num = float(m.group(1))
            mult = m.group(2)
            if mult and mult.upper() == 'K':
                num *= 1000
            elif mult and mult.upper() == 'M':
                num *= 1_000_000
            return int(num)

        emp_df[emp_col] = emp_df[emp_col].apply(parse_employee_count)
        emp_df = emp_df[emp_df[emp_col].notna() & (emp_df[emp_col] > 0)]
        if len(emp_df) > 0:
            generate_chart('top10_total_employee_count',
                create_top10_countries_chart, emp_df, emp_col,
                'Total Employee Count', 'top10_total_employee_count', color='#88CCEE',
                description='Top 10 countries by total employee count across all companies. Employee counts parsed from strings with K/M multipliers.\nOriginal: missing/Economic Plots/StateofMarket/Top 10/Top10_Total Employee Count.png')

    plt.close('all')
    gc.collect()
    print(f"  Top 10 charts: {len(charts_generated) - count_before} generated")

# --- Write .txt description files ---
for chart_name, desc in CHART_DESCRIPTIONS.items():
    txt_path = os.path.join(CHART_OUTPUT_DIR, f'{chart_name}_no_title.txt')
    with open(txt_path, 'w') as f:
        f.write(desc + '\n')

# --- Summary Report ---
print(f"\n{'='*60}")
print(f"Missing Charts Generation Complete")
print(f"{'='*60}")
print(f"Generated: {len(charts_generated)} charts")
print(f"Failed: {len(charts_failed)} charts")
if charts_failed:
    print(f"\nFailed charts:")
    for name, error in charts_failed:
        print(f"  - {name}: {error}")
png_count = len([f for f in os.listdir(CHART_OUTPUT_DIR) if f.endswith('.png')]) if os.path.exists(CHART_OUTPUT_DIR) else 0
print(f"\nOutput directory: {CHART_OUTPUT_DIR}")
print(f"Total PNG files: {png_count}")
print(f"{'='*60}")

In [ ]:
# Cell 8: Global VC Totals Chart (optional)

if SKIP_VC_UPLOAD:
    print("Skipping global VC CSV upload (SKIP_VC_UPLOAD = True).\n")
else:
    print("Upload the global VC totals CSV (e.g., global_vc_totals_2000_to_2025.csv).")
    vc_uploaded = files.upload()
    vc_filenames = list(vc_uploaded.keys())
    assert len(vc_filenames) == 1, f"Expected 1 CSV file, got {len(vc_filenames)}: {vc_filenames}"

    vc_df = pd.read_csv(vc_filenames[0])

    # Validate expected columns
    assert 'Year' in vc_df.columns, \
        f"CSV missing 'Year' column. Columns: {list(vc_df.columns)}"
    VC_AMOUNT_COL = 'Global VC Investment (USD billions)'
    assert VC_AMOUNT_COL in vc_df.columns, \
        f"CSV missing '{VC_AMOUNT_COL}' column. Columns: {list(vc_df.columns)}"

    print(f"Loaded: {vc_filenames[0]} ({len(vc_df)} rows, years {int(vc_df['Year'].min())}-{int(vc_df['Year'].max())})")

    generate_chart('global_vc_totals',
        create_global_vc_funding_chart, vc_df, 2000, 'global_vc_totals',
        description='Global VC investment by year (2000+) from external summary CSV. Not derived from Tracxn company/funding data.\nOriginal: missing/Economic Plots/StateofMarket/007 global_vc_totals_2000_to_2024.png')

    # Write .txt for VC chart (since it runs after Cell 7's txt loop)
    vc_txt = os.path.join(CHART_OUTPUT_DIR, 'global_vc_totals_no_title.txt')
    with open(vc_txt, 'w') as f:
        f.write(CHART_DESCRIPTIONS.get('global_vc_totals', '') + '\n')

    plt.close('all')
    gc.collect()
    print(f"\nGlobal VC chart generated.")

In [ ]:
# Cell 9: Export & Download

zip_filename = 'missing_charts.zip'
zip_count = 0

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for png_file in sorted(os.listdir(CHART_OUTPUT_DIR)):
        if png_file.endswith('.png'):
            filepath = os.path.join(CHART_OUTPUT_DIR, png_file)
            zf.write(filepath, arcname=png_file)
            zip_count += 1

print(f"ZIP created: {zip_filename}")
print(f"Files in ZIP: {zip_count}")
print(f"ZIP size: {os.path.getsize(zip_filename) / 1024 / 1024:.1f} MB")
print(f"\nNote: Colab's filesystem is ephemeral. Download the ZIP before the session ends.")

files.download(zip_filename)
